# Benchmark on Colab: EasyOCR and a page-reading model

Runs the same scoring as the repo (`python -m benchmark.run`) with two more readers. Results are written to your Drive.
Use a GPU runtime: Runtime, Change runtime type, T4 GPU.

Before the first run, upload the benchmark pages to Drive so the folders look like this:

    MyDrive/hindi_benchmark/benchmark_v1/setA_standard/dataset/{train,val,test}/...
                                         setB_hard/dataset/...
                                         setC_ladder/dataset/...
                                         controlA/dataset/...   (optional, the printed control)

In [ ]:
# 1. GPU and Drive
import subprocess, os
print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout or "NO GPU: change the runtime type")
from google.colab import drive
drive.mount("/content/drive")
DATA = "/content/drive/MyDrive/hindi_benchmark/benchmark_v1"
OUT = "/content/drive/MyDrive/hindi_benchmark/results"
BRANCH = "main"        # use "benchmark" if the pull request is not merged yet
SETS = {n: f"{DATA}/{d}/dataset" for n, d in
        {"A": "setA_standard", "B": "setB_hard", "C": "setC_ladder", "P": "controlA"}.items()
        if os.path.isdir(f"{DATA}/{d}/dataset")}
print("sets found:", {k: v.split("/")[-2] for k, v in SETS.items()})

In [ ]:
# 2. Code and libraries
%cd /content
!rm -rf Hindi-OCR-Pipeline
!git clone -q --branch $BRANCH https://github.com/Arraddhyyad/Hindi-OCR-Pipeline.git
%cd /content/Hindi-OCR-Pipeline
!pip install -q rapidfuzz easyocr accelerate bitsandbytes
!pip install -q -U transformers

In [ ]:
# 3. Quick check on ONE page each (read the output before the long runs)
SET_ARGS = " ".join(f"--set {k}={v}" for k, v in SETS.items() if k in ("A",))
!python -m benchmark.run $SET_ARGS --backend easyocr --out /content/try_easy --limit 1
!python -m benchmark.run $SET_ARGS --backend qwen_vl --out /content/try_qwen --limit 1
!head -c 400 /content/try_qwen/preds/*/A/*.txt
!cat /content/try_qwen/summary.md | head -20

In [ ]:
# 4. EasyOCR on everything, damaged and clean copies
SET_ARGS = " ".join(f"--set {k}={v}" for k, v in SETS.items())
!python -m benchmark.run $SET_ARGS --backend easyocr --out "$OUT/easyocr_full" --clean
!cat "$OUT/easyocr_full/summary.md"

In [ ]:
# 5. Page-reading model on an even sample (slow: about a minute or more per dense page)
# 10 pages per set, damaged copies only. Raise SPREAD for more. To try a bigger model use for example:
#   --model Qwen/Qwen3-VL-8B-Instruct --load-4bit
SPREAD = 10
SET_ARGS = " ".join(f"--set {k}={v}" for k, v in SETS.items())
!python -m benchmark.run $SET_ARGS --backend qwen_vl --out "$OUT/qwen_vl_spread" --spread $SPREAD
!cat "$OUT/qwen_vl_spread/summary.md"

Send the two `summary.md` files back. Predictions are saved page by page, so if the session stops, run the cell again and it continues where it left off.
Check `failed` (pages with more than half the characters wrong) next to the median: a model that loops or invents text shows up there.